# SI section 2: The Charles River basin
Each cell prints one sentence of the paragraph on the Charles River basin, or of the caption of Figure S1, and every
number the sentence quotes.
The column SI is the quoted value and the column computed is the value at the precision of the SI.
Data: `python -m fixloc.make.si.charles_basin` for the population structure, and the cluster runs in `sim/` of
`run_network.sh` at $r=1.5$, $2$, and $5$, of `run_network_capped.sh` at $r=1$, and of `run_charles_T.sh` for the mutant
sets at the expected fixation time, summarized by `fixloc.large` (see `src/fixloc/make/si/charles_basin.py`).

In [1]:
import json
from fractions import Fraction

import networkx as nx
import numpy as np

from fixloc.make.si import quote
from fixloc.sim import read_meta

load = lambda f: json.load(open(f))
frac = lambda x: str(Fraction(float(x)).limit_denominator(100))
basin = load('charles_basin.json')
net = load('charles_network.json')
N, root, theta = net['N'], net['outlet'], np.array(net['theta'])
C = np.zeros((N, N))  # C[u, v] is the number of copies of the directed edge u -> v, the weight of placing at v
for u, v in net['edges']:
  C[u, v] += 1
W = C / C.sum(axis=1, keepdims=True)  # W[u, v] = w_uv, the probability that an offspring of u is placed at v
G = nx.Graph([tuple(e) for e in net['edges']])
depth = nx.single_source_shortest_path_length(G, root)
leaves = sorted(v for v in G if G.degree(v) == 1)
near = [v for v in leaves if G.has_edge(v, root)]
watch = load('sim/charles_watch_T.json')
RS = ('1.5', '2.0', '5.0')
run = {r: watch[f'60:{r}'] for r in RS}
run['1.0'] = watch['61:1.0']
last = {r: np.array(x['last']) for r, x in run.items()}
atT = {r: x['at_T'] for r, x in run.items()}
summary = {r: load(f'sim/summary_r{r}.json') for r in run}

## The population structure

In [2]:
quote(
  r'We built the river network of the Charles River basin in Massachusetts from the HydroRIVERS database~\cite{lehner2013global}.',
)
mx, my = basin['mouth']
end = basin['reaches'][root]['end']
km = 111 * np.hypot((end[0] - mx) * np.cos(np.radians(my)), end[1] - my)
print(basin['source'])
print(f"root: vertex {root}, the outlet reach {basin['outlet']}, whose downstream end is {km:.1f} km from the mouth {basin['mouth']}")
quote(r'Each of the reaches of the basin is a vertex ($N=77$).', ('77', len(basin['reaches']), '{}'), ('77', N, '{}'))
print('vertex v is the reach ids[v] of the basin:', net['ids'] == [x['id'] for x in basin['reaches']])
quote(
  r'The graph is a tree rooted at the vertex at the mouth of the river.',
  ('True', nx.is_tree(G), '{}'),
  ('True', net['ids'][root] == basin['outlet'], '{}'),
)
print(G.number_of_nodes(), 'vertices and', G.number_of_edges(), 'edges')
inflow = {x['next_down'] for x in basin['reaches']}
headwaters = [v for v, x in enumerate(basin['reaches']) if x['id'] not in inflow]
quote(r'The leaves of the tree are the headwater reaches.', ('True', leaves == headwaters, '{}'))
print(len(leaves), 'leaves')
index = {rid: v for v, rid in enumerate(net['ids'])}
downstream = {v: index[x['next_down']] for v, x in enumerate(basin['reaches']) if x['next_down']}
quote(
  r'Water flows from the leaves toward the root.',
  ('True', set(downstream) == set(range(N)) - {root}, '{}'),
  ('True', all(depth[d] == depth[v] - 1 for v, d in downstream.items()), '{}'),
)
print('every reach except the root drains (HydroRIVERS NEXT_DOWN) into its neighbor one edge closer to the root')
meet = {frozenset((d, v)) for v, d in downstream.items()}
quote(
  r'Two vertices are joined by an edge in each direction if the two reaches meet.',
  ('True', meet == {frozenset(e) for e in G.edges}, '{}'),
  ('True', all(C[u, v] > 0 and C[v, u] > 0 for u, v in G.edges), '{}'),
)

We built the river network of the Charles River basin in Massachusetts from the HydroRIVERS database [ref].

HydroRIVERS v1.0, North and Central America, Lehner and Grill 2013
root: vertex 0, the outlet reach 70504537, whose downstream end is 0.4 km from the mouth [-71.053, 42.372]
Each of the reaches of the basin is a vertex (N=77).
    SI         77   computed         77   (77)   ok
    SI         77   computed         77   (77)   ok

vertex v is the reach ids[v] of the basin: True
The graph is a tree rooted at the vertex at the mouth of the river.
    SI       True   computed       True   (True)   ok
    SI       True   computed       True   (True)   ok

77 vertices and 76 edges
The leaves of the tree are the headwater reaches.
    SI       True   computed       True   (True)   ok

39 leaves
Water flows from the leaves toward the root.
    SI       True   computed       True   (True)   ok
    SI       True   computed       True   (True)   ok

every reach except the root drains (Hydr

In [3]:
pairs = list(zip(*np.nonzero(C)))
down = sorted({float(C[u, v]) for u, v in pairs if depth[v] < depth[u]})
up = sorted({float(C[u, v]) for u, v in pairs if depth[v] > depth[u]})
weights = lambda s: ', '.join(f'{x:g}' for x in s)
quote(r'The two directions have different weights.', ('True', all(C[u, v] != C[v, u] for u, v in G.edges), '{}'))
quote(
  r'An offspring is placed downstream, toward the root, with weight $2$ and upstream, away from the root, with weight $1$.',
  ('2', down, weights),
  ('1', up, weights),
)
quote(r'We give the downstream direction the larger weight because the current carries offspring downstream.')
print('the root has', G.degree(root), 'neighbors and is not a leaf')
far = [v for v in leaves if v not in near]
fracs = lambda s: ', '.join(frac(x) for x in s)
quote(
  r'Every leaf has temperature $1/4$, except the leaf adjacent to the root with temperature $1/2$.',
  ('1/4', sorted(set(theta[far].tolist())), fracs),
  ('1/2', theta[near].tolist(), fracs),
)
print(len(far), 'leaves have temperature 1/4; the leaf adjacent to the root is vertex', near[0])
inner = [v for v in G if v not in leaves]
quote(r'Every other vertex has temperature at least $5/4$.', ('5/4', theta[inner].min(), frac))
print('the root, vertex', root, 'has temperature', frac(theta[root]))
print('temperatures of the other vertices:', fracs(sorted(set(theta[inner].tolist()))))
dev = np.abs(W.sum(axis=0) - theta).max()
print(f'largest difference between the stored temperature theta_v and sum_u w_uv: {dev:.1e}')
assert dev < 1e-12

The two directions have different weights.
    SI       True   computed       True   (True)   ok

An offspring is placed downstream, toward the root, with weight 2 and upstream, away from the root, with weight 1.
    SI          2   computed          2   ([2.0])   ok
    SI          1   computed          1   ([1.0])   ok

We give the downstream direction the larger weight because the current carries offspring downstream.

the root has 2 neighbors and is not a leaf
Every leaf has temperature 1/4, except the leaf adjacent to the root with temperature 1/2.
    SI        1/4   computed        1/4   ([0.25])   ok
    SI        1/2   computed        1/2   ([0.5])   ok

38 leaves have temperature 1/4; the leaf adjacent to the root is vertex 1
Every other vertex has temperature at least 5/4.
    SI        5/4   computed        5/4   (1.25)   ok

the root, vertex 0 has temperature 3/2
temperatures of the other vertices: 5/4, 3/2, 7/4, 2, 9/4
largest difference between the stored temperature the

## Simulated fixation locations at $r=1.5$, $2$, and $5$

In [4]:
quote(r'The graph has too many vertices for the exact computation of \Cref{sec:computation}.')
print(f'the exact computation runs on all 2^{N} = {2.0 ** N:.1e} sets of mutants')
quote(
  r'We therefore estimated the fixation location probabilities from $4{,}000$ simulated fixation events from each initial vertex at $r=1.5$, $2$, and $5$.',
  *[('4,000', run[r]['fixations_main'] / N, '{:,.0f}') for r in RS],
)
print('fixation events of the main runs:', {r: run[r]['fixations_main'] for r in RS}, end=' ')
print('= 77 x 4,000, and sweepsim stops an initial vertex at 4,000')
weight = {}
for r in run:
  _, _, runs_u, fixes_u, _ = read_meta(f'sim/main_r{r}.bin')
  q = fixes_u / runs_u
  weight[r] = q / q.sum()
quote(
  r'Each initial vertex has a weight proportional to the estimated fixation probability, the fraction of simulated trajectories from the initial vertex ending in fixation.',
  *[('True', bool(np.allclose(weight[r], run[r]['pi'])), '{}') for r in run],
)
print('the weights pi of fixloc.large.location_law are the fixation events divided by the trajectories of each initial vertex, normalized to sum 1')
share = {r: last[r][leaves].sum() for r in run}
quote(
  r'For a uniformly random initial vertex, the last wild type occupies a leaf with probability $0.957$, $0.974$, and $0.994$ at $r=1.5$, $2$, and $5$, respectively (\Cref{fig:charles}).',
  *[(si, share[r], '{:.3f}') for r, si in zip(RS, ('0.957', '0.974', '0.994'))],
)
hot = np.flatnonzero(theta > 1)
quote(
  r'Every vertex with temperature above $1$ has an estimated fixation location probability below $0.012$.',
  *[('< 0.012', last[r][hot].max(), lambda x: '< 0.012' if x < 0.012 else f'{x:.4f}') for r in RS],
)
for r in run:
  v = hot[np.argmax(last[r][hot])]
  print(f'r = {r}: largest fixation location probability at a vertex with temperature above 1 is {last[r][v]:.4f}, at vertex {v}')

The graph has too many vertices for the exact computation of [ref].

the exact computation runs on all 2^77 = 1.5e+23 sets of mutants
We therefore estimated the fixation location probabilities from 4,000 simulated fixation events from each initial vertex at r=1.5, 2, and 5.
    SI      4,000   computed      4,000   (4000)   ok
    SI      4,000   computed      4,000   (4000)   ok
    SI      4,000   computed      4,000   (4000)   ok

fixation events of the main runs: {'1.5': 308000, '2.0': 308000, '5.0': 308000} = 77 x 4,000, and sweepsim stops an initial vertex at 4,000
Each initial vertex has a weight proportional to the estimated fixation probability, the fraction of simulated trajectories from the initial vertex ending in fixation.
    SI       True   computed       True   (True)   ok
    SI       True   computed       True   (True)   ok
    SI       True   computed       True   (True)   ok
    SI       True   computed       True   (True)   ok

the weights pi of fixloc.large.locati

In [5]:
D = max(depth[v] for v in leaves)
farthest = sorted(v for v in leaves if depth[v] == D)
top2 = {r: sorted(np.argsort(-last[r])[:2].tolist()) for r in run}
vertices = lambda s: ', '.join(map(str, s))
quote(
  r'At $r=1.5$, $2$, and $5$, the two likeliest fixation locations are the two leaves farthest from the root.',
  *[(vertices(farthest), top2[r], vertices) for r in RS],
)
print('leaves farthest from the root:', farthest, f'at {D} edges; the next leaves are at', sorted({depth[v] for v in leaves})[-2], 'edges')
swap = np.arange(N)
swap[farthest] = farthest[::-1]
quote(
  r'The two leaves have the same downstream neighbor and therefore equal fixation location probabilities.',
  ('True', len({downstream[v] for v in farthest}) == 1, '{}'),
  ('True', np.array_equal(C[np.ix_(swap, swap)], C), '{}'),
)
print('vertices', vertices(farthest), 'drain into vertex', downstream[farthest[0]], end='; ')
print('exchanging the two leaves leaves every weight unchanged, so the true probabilities are equal')
for r in RS:
  print(f'r = {r}: simulated fixation location probabilities of vertices 75 and 76: {last[r][75]:.4f} and {last[r][76]:.4f}')

At r=1.5, 2, and 5, the two likeliest fixation locations are the two leaves farthest from the root.
    SI     75, 76   computed     75, 76   ([75, 76])   ok
    SI     75, 76   computed     75, 76   ([75, 76])   ok
    SI     75, 76   computed     75, 76   ([75, 76])   ok

leaves farthest from the root: [75, 76] at 27 edges; the next leaves are at 26 edges
The two leaves have the same downstream neighbor and therefore equal fixation location probabilities.
    SI       True   computed       True   (True)   ok
    SI       True   computed       True   (True)   ok

vertices 75, 76 drain into vertex 72; exchanging the two leaves leaves every weight unchanged, so the true probabilities are equal
r = 1.5: simulated fixation location probabilities of vertices 75 and 76: 0.1712 and 0.1714
r = 2.0: simulated fixation location probabilities of vertices 75 and 76: 0.1743 and 0.1756
r = 5.0: simulated fixation location probabilities of vertices 75 and 76: 0.1612 and 0.1598


The best vertex to sample at time step $t$ is the vertex $w$ with the smallest $m_w(t)$, the probability of a mutant at
$w$ at time step $t$ conditional on fixation. The sampling time is the expected fixation time $T$ for a uniformly random
initial vertex, conditional on fixation, from the main run (`fixloc.large.expected_time`). `run_charles_T.sh` records
the mutant sets at time step round($T$) from $4{,}000$ fixation events per initial vertex, with at most $10^6$ runs per
initial vertex at $r=1$, and `fixloc.large.watch_at_T` gives $m_w(T)$ with standard errors stratified by the initial
vertex. A symmetry of the tree exchanges two leaves with the same downstream neighbor. Two such leaves have equal
fixation location probabilities and equal $m_w(T)$, so their order in a simulation is sampling noise, and the SI names
the pair.

In [6]:
ranks = lambda a: sorted([a['rank'], a['rank_second']])
pairs_of = lambda a: sorted([a['best'], a['second']])
quote(
  r'At the expected fixation time, the best vertices to sample are the two likeliest fixation locations at $r=1.5$, $2$, and $5$.',
  *[('1, 2', ranks(atT[r]), vertices) for r in RS],
)
for r in RS:
  a = atT[r]
  print(f"r = {r}: T = {run[r]['T']:,.1f} (summary mean_duration equal: {run[r]['T'] == summary[r]['mean_duration']}),",
        f"checkpoint {a['t']:,}, f_T = {a['f']:.3f}, {a['ck_fixations']:,} fixation events")
  print(f"   smallest m_w(T) at vertices {vertices(pairs_of(a))}, the farthest leaves: {pairs_of(a) == farthest};",
        f"m = {a['m_best']:.4f} and {a['m_second']:.4f}, {a['z_gap']:.1f} standard errors apart")
  print(f"   next vertex {a['third']}, m = {a['m_third']:.4f}, {a['z_third']:.1f} standard errors above the pair")

At the expected fixation time, the best vertices to sample are the two likeliest fixation locations at r=1.5, 2, and 5.
    SI       1, 2   computed       1, 2   ([1, 2])   ok
    SI       1, 2   computed       1, 2   ([1, 2])   ok
    SI       1, 2   computed       1, 2   ([1, 2])   ok

r = 1.5: T = 46,094.9 (summary mean_duration equal: True), checkpoint 46,095, f_T = 0.585, 308,000 fixation events
   smallest m_w(T) at vertices 75, 76, the farthest leaves: True; m = 0.6056 and 0.6056, 0.3 standard errors apart
   next vertex 72, m = 0.6084, 20.3 standard errors above the pair
r = 2.0: T = 19,220.5 (summary mean_duration equal: True), checkpoint 19,221, f_T = 0.575, 308,000 fixation events
   smallest m_w(T) at vertices 75, 76, the farthest leaves: True; m = 0.6160 and 0.6162, 0.8 standard errors apart
   next vertex 72, m = 0.6232, 35.6 standard errors above the pair
r = 5.0: T = 5,973.9 (summary mean_duration equal: True), checkpoint 5,974, f_T = 0.574, 308,000 fixation events
   s

## Neutral evolution

In [7]:
head = open('sim/main_r1.0.bin.meta').readline().split()
K = int(head[head.index('K') + 1])
_, _, runs, fixes, _ = read_meta('sim/main_r1.0.bin')
power = lambda x: f'10^{np.log10(x):.0f}' if np.log10(x) % 1 == 0 else f'{x:,}'
quote(
  r'At $r=1$, we simulated from each initial vertex until $4{,}000$ fixation events or $10^6$ trajectories, because a mutant near the root rarely reaches fixation.',
  ('4,000', K, '{:,}'),
  ('10^6', runs.max(), power),
)
print(int((fixes == K).sum()), 'initial vertices reached 4,000 fixation events', end=', ')
print(int(((runs == runs.max()) & (fixes < K)).sum()), 'stopped at 10^6 trajectories')

# Exact neutral fixation probabilities (Allen et al. 2015, PLoS Comput Biol 11:e1004108, Eqs. 1 and 2):
# theta_u rho_u = sum_v w_uv rho_v for every vertex u, and sum_u rho_u = 1.
A = np.vstack([np.diag(theta) - W, np.ones(N)])
b = np.r_[np.zeros(N), 1.0]
rho = np.linalg.lstsq(A, b, rcond=None)[0]
print(f'largest residual of the linear system: {np.abs(A @ rho - b).max():.1e}')
none = np.flatnonzero(fixes == 0)
below = lambda x: '< 10^-5' if x < 1e-5 else f'{x:.3g}'
quote(
  r'From each of $24$ initial vertices, none of the $10^6$ simulated trajectories reached fixation.',
  ('24', len(none), '{}'),
  ('10^6', runs[none].min(), power),
  ('10^6', runs[none].max(), power),
)
print('initial vertices without a simulated fixation event:', none.tolist())
print()
# For a uniformly random initial vertex, p_u = rho_u / sum(rho) is the probability of the initial vertex u given fixation.
eps = rho[none].sum() / rho.sum()
quote(
  r'By the exact neutral fixation probabilities~\cite{allen2015molecular}, the initial vertex is one of the $24$ vertices with probability below $10^{-5}$, conditional on fixation.',
  ('24', len(none), '{}'),
  ('< 10^-5', eps, below),
)
# With L(v|u) the fixation location probability from the initial vertex u, the fixation location probability is
# L(v) = sum_u p_u L(v|u). Without the initial vertices in none, the fixation location probability is
# L'(v) = sum_{u not in none} p_u L(v|u) / (1 - eps). The difference L(v) - L'(v) equals the first term
# sum_{u in none} p_u L(v|u) minus the second term eps / (1 - eps) sum_{u not in none} p_u L(v|u).
# Each term lies between 0 and eps, so |L(v) - L'(v)| <= eps for every vertex v.
quote(
  r'Omitting the $24$ initial vertices therefore changes every fixation location probability by less than $10^{-5}$.',
  ('24', len(none), '{}'),
  ('< 10^-5', eps, below),
)
print(f'probability of an initial vertex without a simulated fixation event, given fixation: {eps:.3g}')
print(f'exact fixation probability {rho[root]:.2g} at the root, {rho[near[0]]:.2g} at the leaf adjacent to the root,', end=' ')
print(f'{rho.max():.3f} at vertex {rho.argmax()}')
expected = runs * rho
z = (fixes - expected) / np.sqrt(expected * (1 - rho))
print(f'simulated against exact fixation events: chi-square {np.sum(z ** 2):.1f} over {N} initial vertices,', end=' ')
print(f'largest |z| {np.abs(z).max():.1f} at vertex {np.abs(z).argmax()} ({fixes[np.abs(z).argmax()]} events, {expected[np.abs(z).argmax()]:.1f} expected)')

At r=1, we simulated from each initial vertex until 4,000 fixation events or 10^6 trajectories, because a mutant near the root rarely reaches fixation.
    SI      4,000   computed      4,000   (4000)   ok
    SI       10^6   computed       10^6   (1000000)   ok

18 initial vertices reached 4,000 fixation events, 59 stopped at 10^6 trajectories
largest residual of the linear system: 6.7e-16
From each of 24 initial vertices, none of the 10^6 simulated trajectories reached fixation.
    SI         24   computed         24   (24)   ok
    SI       10^6   computed       10^6   (1000000)   ok
    SI       10^6   computed       10^6   (1000000)   ok

initial vertices without a simulated fixation event: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 26]

By the exact neutral fixation probabilities [ref], the initial vertex is one of the 24 vertices with probability below 10^-5, conditional on fixation.
    SI         24   computed         24   (24)   ok
   

In [8]:
quote(
  r'For a uniformly random initial vertex, the last wild type occupies a leaf with probability $0.956$ at $r=1$.',
  ('0.956', share['1.0'], '{:.3f}'),
)
quote(
  r'The likeliest fixation location at $r=1$ is the leaf adjacent to the root, with fixation location probability $0.365$.',
  (str(near[0]), int(np.argmax(last['1.0'])), '{}'),
  ('0.365', last['1.0'].max(), '{:.3f}'),
)
a = atT['1.0']
pair = pairs_of(a)
quote(
  r'At the expected fixation time, the best vertices to sample are the third and fourth likeliest fixation locations at $r=1$, two leaves with the same downstream neighbor.',
  ('3, 4', ranks(a), vertices),
  ('True', all(v in leaves for v in pair) and downstream[pair[0]] == downstream[pair[1]], '{}'),
)
print(f"r = 1.0: T = {run['1.0']['T']:,.1f} (summary mean_duration equal: {run['1.0']['T'] == summary['1.0']['mean_duration']}),",
      f"checkpoint {a['t']:,}, f_T = {a['f']:.3f}, {a['ck_fixations']:,} fixation events")
print(f"   smallest m_w(T) at vertices {vertices(pair)}, which drain into vertex {downstream[pair[0]]};",
      f"m = {a['m_best']:.4f} and {a['m_second']:.4f}, {a['z_gap']:.1f} standard errors apart")
print(f"   next vertex {a['third']}, m = {a['m_third']:.4f}, {a['z_third']:.1f} standard errors above the pair")
print(f"   the likeliest fixation location, vertex {a['likeliest']}, has m = {a['m_likeliest']:.4f},",
      f"{a['z_likeliest_minus_best']:.1f} standard errors above the pair")

For a uniformly random initial vertex, the last wild type occupies a leaf with probability 0.956 at r=1.
    SI      0.956   computed      0.956   (0.956211)   ok

The likeliest fixation location at r=1 is the leaf adjacent to the root, with fixation location probability 0.365.
    SI          1   computed          1   (1)   ok
    SI      0.365   computed      0.365   (0.365206)   ok

At the expected fixation time, the best vertices to sample are the third and fourth likeliest fixation locations at r=1, two leaves with the same downstream neighbor.
    SI       3, 4   computed       3, 4   ([3, 4])   ok
    SI       True   computed       True   (True)   ok

r = 1.0: T = 105,876.8 (summary mean_duration equal: True), checkpoint 105,877, f_T = 0.573, 94,549 fixation events
   smallest m_w(T) at vertices 19, 20, which drain into vertex 16; m = 0.6683 and 0.6687, 1.0 standard errors apart
   next vertex 16, m = 0.6714, 6.3 standard errors above the pair
   the likeliest fixation location,

## Caption of Figure S1

In [9]:
quote(
  r'Offspring are placed downstream with weight $2$ and upstream with weight $1$.',
  ('2', down, weights),
  ('1', up, weights),
)
quote(
  r'\textbf{b,} Fixation location probability at $r=2$ for a uniformly random initial vertex, conditional on fixation, estimated from $4{,}000$ simulated fixation events from each initial vertex.',
  ('4,000', run['2.0']['fixations_main'] / N, '{:,.0f}'),
)
f = load('../../figS1/b/data.json')
print('the data of Figure S1b are these probabilities at r = 2:', f['r'] == 2.0 and f['P_last'] == run['2.0']['last'])

Offspring are placed downstream with weight 2 and upstream with weight 1.
    SI          2   computed          2   ([2.0])   ok
    SI          1   computed          1   ([1.0])   ok

b, Fixation location probability at r=2 for a uniformly random initial vertex, conditional on fixation, estimated from 4,000 simulated fixation events from each initial vertex.
    SI      4,000   computed      4,000   (4000)   ok

the data of Figure S1b are these probabilities at r = 2: True


## Summary by fitness

In [10]:
print(f"{'r':>4} {'leaf share':>10} {'max theta>1':>11} {'two likeliest':>13} {'likeliest':>9} {'P':>6} {'T':>9} {'best pair':>9} {'ranks':>5}")
for r in ('1.0',) + RS:
  L, a = last[r], atT[r]
  print(
    f'{r:>4} {share[r]:10.3f} {L[hot].max():11.4f} {vertices(top2[r]):>13} {int(np.argmax(L)):9d} {L.max():6.3f}',
    f"{run[r]['T']:9,.0f} {vertices(pairs_of(a)):>9} {vertices(ranks(a)):>5}",
  )

   r leaf share max theta>1 two likeliest likeliest      P         T best pair ranks
 1.0      0.956      0.0088          1, 4         1  0.365   105,877    19, 20  3, 4
 1.5      0.957      0.0117        75, 76        76  0.171    46,095    75, 76  1, 2
 2.0      0.974      0.0075        75, 76        76  0.176    19,221    75, 76  1, 2
 5.0      0.994      0.0016        75, 76        75  0.161     5,974    75, 76  1, 2
